In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
from bridgit.players import Arena, RandomPlayer, MCTSPlayer
from bridgit.ai import NetWrapper, BridgitNet
from bridgit.ai.self_play import batched_self_play
from bridgit.config import BoardConfig, MCTSConfig, NeuralNetConfig
from bridgit import Visualizer

In [ ]:
board_config = BoardConfig(size=4)
net= BridgitNet(board_config=board_config, net_config=NeuralNetConfig())

checkpoint_path = "../trainings/run_2026-03-19_003409/iteration_003/post_training.pt"
trained_wrapper = NetWrapper(checkpoint_path)

player_1 = MCTSPlayer(net_wrapper=NetWrapper(net), mcts_config=MCTSConfig(num_simulations=5000), name="MCTS_untrained", temperature=0)
player_2 = MCTSPlayer(net_wrapper=trained_wrapper, mcts_config=MCTSConfig(num_simulations=5000), name="MCTS_trained", temperature=0)

arena = Arena(player_1, player_2, board_config)

In [ ]:
# Play a single game
record = arena.play_game(verbose=True)
print(record.summary())

In [ ]:
Visualizer.visualize_game(record)

In [ ]:
# Play multiple games (sequential)
collection = arena.play_games(num_games=12, verbose=True, swap_players=True)
print(f"Results over {len(collection)} games: {collection.scores}")

In [ ]:
Visualizer.visualize_game(collection.game_records[0])

## Batched Self-Play

Runs N games concurrently in a single process, batching all neural net calls into one GPU forward pass per MCTS simulation step.

In [ ]:
import time

wrapper = NetWrapper(BridgitNet(board_config=board_config))
mcts_config = MCTSConfig(num_simulations=200)
NUM_GAMES = 20

# Sequential (one game at a time)
player = MCTSPlayer(net_wrapper=wrapper, mcts_config=mcts_config, name="seq", temperature=1.0)
seq_arena = Arena(player, player, board_config)
t0 = time.time()
seq_records = seq_arena.play_games(num_games=NUM_GAMES, verbose=True)
t_seq = time.time() - t0
print(f"\nSequential: {t_seq:.1f}s ({NUM_GAMES/t_seq:.1f} games/s)")

# Batched (all games concurrently)
t0 = time.time()
batch_records = batched_self_play(
    wrapper, board_config, mcts_config,
    num_games=NUM_GAMES, batch_size=NUM_GAMES, temperature=1.0, verbose=True,
)
t_batch = time.time() - t0
print(f"\nBatched: {t_batch:.1f}s ({NUM_GAMES/t_batch:.1f} games/s)")
print(f"Speedup: {t_seq/t_batch:.1f}x")

In [ ]:
# Visualize a game from the batched run
Visualizer.visualize_game(batch_records.game_records[0])